In [23]:
import deeptrack as dt
from deeptrack.backend import xp
config = dt.backend.config

# What is xp?

`xp` is a proxy that provides a unified interface to array operations between different
computational backends. We primarily use it to provide a unified interface to numpy and torch,
but it also unifies operations between different array libraries, such as cupy and jax. 

Syntax is generally unified to be similar to numpy, but there are some differences.
Primarily, operations may accept some additional arguments (such as device).

`xp` is intended for internal use. Users may use `xp` when writing custom features, but
are not forced to do so (only if they want to be compatible with all backends).


In [24]:
import numpy as np

out_with_numpy = np.sum(np.random.randn(100, 100), keepdims=True)
out_with_xp = xp.sum(xp.random.randn(100, 100), keepdims=True)

print(f"type(out_with_numpy): {type(out_with_numpy)}")
print(f"type(out_with_xp): {type(out_with_xp)}")

type(out_with_numpy): <class 'numpy.ndarray'>
type(out_with_xp): <class 'numpy.ndarray'>


## Changing the backend

The `config.set_backend()` function allows you to switch between computational backends.
When you change the backend, all subsequent `xp` operations will use the specified backend.
This is useful for writing code that can run on different backends without changing the implementation.
In the example below, we use the same `xp` syntax to perform operations with both numpy and torch backends.




In [25]:
config.set_backend("numpy")
out_with_numpy = xp.sum(xp.random.randn(100, 100), keepdims=True)
config.set_backend("torch")
out_with_torch = xp.sum(xp.random.randn(100, 100), keepdims=True)

print(f"type(out_with_numpy): {type(out_with_numpy)}")
print(f"type(out_with_torch): {type(out_with_torch)}")

type(out_with_numpy): <class 'numpy.ndarray'>
type(out_with_torch): <class 'torch.Tensor'>


Note that torch syntax for the same expression is different: (`keepdim`	vs. `keepdims`, and `torch.randn` vs. `random.randn`). Moreover, for torch `keepdim`, you also need to specify the dimensions you want to keep. This shows the value of `xp` when keeping interoperability between backends!

In [26]:
import torch 
# torch.sum(torch.randn(100, 100), keepdim=True) # This will fail

out = torch.sum(torch.randn(100, 100), dim=(0, 1), keepdim=True)
print(f"type(out): {type(out)}")


type(out): <class 'torch.Tensor'>


## Using xp in custom features

The following example demonstrates how to use the `xp` module within a custom DeepTrack feature.
By using `xp` instead of directly calling numpy or torch functions, your feature will automatically
work with the currently selected backend, making your code more flexible and backend-agnostic.


In [27]:
class Zeros(dt.Feature):

    # Because this feature takes no inputs, it is not distributed
    __distributed__ = False

    def __init__(self, shape: tuple[int, int]):
        super().__init__(shape=shape)

    def get(self, _, shape: tuple[int, int], **kwargs):
        return xp.zeros(shape)

In the example below, we create a simple `Zeros` feature that uses `xp.zeros()` to generate arrays of zeros. 
This feature will automatically work with the current backend (numpy or torch) without any changes to the code.

Notice how we can switch backends globally using `config.set_backend()`

In [42]:
config.set_backend("numpy")
feature_a = Zeros((100, 100))
out_with_numpy = feature_a()

config.set_backend("torch")
feature_b = Zeros((100, 100))
out_with_torch = feature_b()

print(type(out_with_numpy), "expected numpy.ndarray")
print(type(out_with_torch), "expected torch.Tensor")

<class 'numpy.ndarray'> expected numpy.ndarray
<class 'torch.Tensor'> expected torch.Tensor



DeepTrack features maintain their backend configuration even when the global backend changes. This means that once a feature is created with a specific backend, it will continue to use that backend for all operations, regardless of subsequent changes to the global configuration.

The following code demonstrates this behavior by creating a feature with the numpy backend and then changing the global backend to torch. Notice how the feature still produces numpy arrays despite the global backend change.


In [29]:
config.set_backend("numpy")
feature_a = Zeros((100, 100))
out_before_change = feature_a()

config.set_backend("torch")
# we changed the global config, but the feature should still use the old backend
# since it was created before the global config was changed
out_after_change = feature_a()

print(type(out_before_change), "expected numpy.ndarray")
print(type(out_after_change), "expected torch.ndarray")

<class 'numpy.ndarray'> expected numpy.ndarray
<class 'numpy.ndarray'> expected torch.ndarray



DeepTrack features also provide methods to explicitly override their backend, regardless of the global configuration. 

You can use `.torch()` to force a feature to use PyTorch, or `.numpy()` to force it to use NumPy. This is particularly useful when you need specific backend functionality for certain operations while maintaining a different global backend.

The following code demonstrates how to override the backend for individual features. Notice how the output types match the explicitly specified backends, not the global configuration.


In [39]:
config.set_backend("numpy")
feature_a = Zeros((100, 100)).torch()
out_a = feature_a()

config.set_backend("torch")
feature_b = Zeros((100, 100)).numpy()
out_b = feature_b()

print(type(out_a), "expected torch.Tensor")
print(type(out_b), "expected numpy.ndarray")


<class 'torch.Tensor'> expected torch.Tensor
<class 'numpy.ndarray'> expected numpy.ndarray


## Chaining features

When chaining features with different backends, you need to be careful about compatibility. DeepTrack features pass data from one feature to the next, and if the features use different backends, you may encounter errors. I propose that we do not automatically convert between backends when chaining features.

The following code demonstrates how to create a feature pipeline using the `>>` operator to chain features together. Note how the backend of the first feature in the chain determines the backend of the entire pipeline's output.


In [41]:
class Sum(dt.Feature):

    def __init__(self, axis: int | None = None, keepdims: bool = False):
        super().__init__(axis=axis, keepdims=keepdims)

    def get(
        self,
        image: np.ndarray | torch.Tensor,
        axis: int | None = None,
        keepdims: bool = False,
        **kwargs,
    ):
        return xp.sum(image, axis=axis, keepdims=keepdims)


The following code demonstrates that the backend of chaiend features correctly uses
the intended backend.


In [32]:
config.set_backend("numpy")
feature = Zeros((100, 100)) >> Sum()
out = feature()

config.set_backend("torch")
feature2 = Zeros((100, 100)) >> Sum()
out2 = feature2()

print(type(out), "expected numpy.ndarray")
print(type(out2), "expected torch.Tensor")

<class 'numpy.float64'> expected numpy.ndarray
<class 'torch.Tensor'> expected torch.Tensor


Here, we see that calling `feature.torch()` applies to the entire pipeline.


In [44]:
config.set_backend("numpy")
feature = Zeros((100, 100)) >> Sum()
feature.torch()
out = feature()
print(type(out), "expected torch.Tensor")

<class 'torch.Tensor'> expected torch.Tensor


The following code demonstrates how chaining features with different backends can lead to compatibility issues. When a feature with one backend tries to process data from a feature with a different backend, a TypeError occurs. This is because DeepTrack does not automatically convert between backends when chaining features.


In [45]:
config.set_backend("numpy")
zeros = Zeros((100, 100)).torch()
summer = Sum().numpy()
feature = zeros >> summer
try:
    out = feature()
except TypeError as e:
    print("Expected TypeError since torch tensor was passed to Sum:")
    print(e)

Expected TypeError since torch tensor was passed to Sum:
sum() received an invalid combination of arguments - got (axis=NoneType, keepdims=bool, out=NoneType, ), but expected one of:
 * (*, torch.dtype dtype = None)
 * (tuple of ints dim, bool keepdim = False, *, torch.dtype dtype = None)
 * (tuple of names dim, bool keepdim = False, *, torch.dtype dtype = None)



## Manually converting to right array type

When working with multiple backends, you may need to manually convert between array types. The following code demonstrates a simple feature that converts any input to a NumPy array, which can be useful when you need to ensure compatibility between features using different backends. This approach provides explicit control over data type conversion in your processing pipeline.



In [46]:
class AsNumpy(dt.Feature):
    def get(self, image: np.ndarray | torch.Tensor, **kwargs):
        # note that we use np.asarray, not np.array.
        return np.asarray(image)

Here, we use the `AsNumpy` feature to convert the torch Zeros output to a NumPy array, which can then be processed by the `Sum` feature.


In [47]:
zeros = Zeros((100, 100)).torch()
converter = AsNumpy()
summer = Sum().numpy()
feature = zeros >> converter >> summer
out = feature()
print(type(out), "expected numpy.ndarray")

<class 'numpy.float32'> expected numpy.ndarray


## Dispatching to the correct backend

Sometimes, you may need specific code paths for different backends.
The following example demonstrates how to dispatch to the correct backend using the `array_api_compat` library.


In [48]:
import array_api_compat as apc

class DispatchExample(dt.Feature):

    def foo(self, image: np.ndarray | torch.Tensor, **kwargs):
        if apc.is_numpy_array(image):
            return self.foo_numpy(image)
        elif apc.is_torch_array(image):
            return self.foo_torch(image)
        else:
            raise TypeError(
                f"Expected numpy.ndarray or torch.Tensor, got {type(image)}"
            )

    def foo_numpy(self, image: np.ndarray, **kwargs):
        print("Called numpy version")
        return image

    def foo_torch(self, image: torch.Tensor, **kwargs):
        print("Called torch version")
        return image

    def get(self, image: np.ndarray | torch.Tensor, **kwargs):
        return self.foo(image)


In [49]:

feature = Zeros((100, 100)) >> DispatchExample()
feature.torch()
out = feature(zeros)

feature.numpy()
out = feature(zeros)


Called torch version
Called numpy version


## Compatability with `Image` and `properties`

xp and Image are compatible, and properties are preserved.

In [57]:
feature = Zeros((100, 100)) >> Sum()
feature.store_properties()
feature.torch()

x = feature()
x.properties, type(x._value)

([{'shape': (100, 100), 'name': 'Zeros'},
  {'axis': None, 'keepdims': False, 'name': 'Sum'},
  {'name': 'Chain'}],
 torch.Tensor)